In [1]:
import torch
import time
import numpy as np
from tqdm import tqdm
import psutil
import subprocess
import platform
import re


# 粗略的 FP32 理论峰值 (TFLOPS)，仅用于估算利用率
PEAK_FP32_TFLOPS = {
    'm1 ultra': 20.8, 'm1 max': 10.4, 'm1 pro': 5.2, 'm1': 2.6,
    'm2 ultra': 27.2, 'm2 max': 13.6, 'm2 pro': 6.8, 'm2': 3.6,
    'm3 ultra': 28.4, 'm3 max': 14.2, 'm3 pro': 7.1, 'm3': 3.5,
    'm4 ultra': 32.8, 'm4 max': 16.4, 'm4 pro': 8.2, 'm4': 4.1,
}


class MacGPU_Ultimate_Test:
    def __init__(self):
        if torch.backends.mps.is_available():
            self.device = torch.device('mps')
            self.backend = 'mps'
        else:
            self.device = torch.device('cpu')
            self.backend = 'cpu'
            print("⚠️  MPS 不可用，将使用 CPU 运行（性能会非常低）")

        self.chip_name = self._get_chip_name()
        vm = psutil.virtual_memory()
        self.total_memory = vm.total / 1e9
        self.available_memory = vm.available / 1e9
        self.peak_fp32 = self._estimate_peak_fp32()

        # powermetrics 需要 sudo -n 免密，失败则自动关闭监控
        self._powermetrics_available = (self.backend == 'mps')

        print(f"🚀 Mac GPU 终极性能测试")
        print(f"芯片: {self.chip_name}")
        print(f"后端: {self.backend}")
        print(f"总内存: {self.total_memory:.1f} GB")
        print(f"可用内存: {self.available_memory:.1f} GB")
        if self.peak_fp32:
            print(f"FP32 理论峰值(估算): {self.peak_fp32:.1f} TFLOPS")
        print("=" * 70)

    def _get_chip_name(self):
        try:
            out = subprocess.run(['sysctl', '-n', 'machdep.cpu.brand_string'],
                                 capture_output=True, text=True, timeout=5)
            name = out.stdout.strip()
            if name:
                return name
        except Exception:
            pass
        return platform.processor() or 'Apple Silicon'

    def _estimate_peak_fp32(self):
        name = self.chip_name.lower()
        for key in sorted(PEAK_FP32_TFLOPS.keys(), key=len, reverse=True):
            if key in name:
                return PEAK_FP32_TFLOPS[key]
        return None

    def _sync(self):
        if self.backend == 'mps':
            torch.mps.synchronize()

    def _mem_allocated_gb(self):
        if self.backend == 'mps':
            return torch.mps.current_allocated_memory() / 1e9
        return 0.0

    def _empty_cache(self):
        if self.backend == 'mps':
            torch.mps.empty_cache()

    def get_gpu_stats(self):
        """返回 (功耗W, 利用率%)。powermetrics 需要 sudo 免密，否则返回 (0, 0)。"""
        if not self._powermetrics_available:
            return 0.0, 0.0
        try:
            result = subprocess.run(
                ['sudo', '-n', 'powermetrics', '-n', '1', '-i', '100', '-s', 'gpu_power'],
                capture_output=True, text=True, timeout=5, stdin=subprocess.DEVNULL
            )
            if result.returncode != 0:
                self._powermetrics_available = False
                return 0.0, 0.0
            out = result.stdout
            util_m = re.search(r'GPU (?:HW )?active residency:\s*([\d.]+)%', out)
            power_m = re.search(r'GPU Power:\s*([\d.]+)\s*mW', out)
            util = float(util_m.group(1)) if util_m else 0.0
            power = float(power_m.group(1)) / 1000.0 if power_m else 0.0
            return power, util
        except Exception:
            self._powermetrics_available = False
            return 0.0, 0.0

    def find_optimal_matrix_size(self):
        # 统一内存，保留 50% 给系统
        available_bytes = psutil.virtual_memory().available * 0.5
        max_size = int(np.sqrt(available_bytes / (3 * 4)))
        optimal_size = (max_size // 64) * 64
        optimal_size = min(optimal_size, 8192)  # Mac 上更保守
        print(f"建议矩阵大小: {optimal_size}x{optimal_size}")
        return optimal_size

    def test_matmul_batch(self, size, batch_size=8, duration=120):
        print(f"\n📊 批处理矩阵乘法测试")
        print(f"矩阵大小: {size}x{size}, 批大小: {batch_size}, 持续时间: {duration}s")
        print("-" * 70)

        a = torch.randn(batch_size, size, size, dtype=torch.float32, device=self.device)
        b = torch.randn(batch_size, size, size, dtype=torch.float32, device=self.device)
        print(f"输入矩阵显存: {(batch_size * size * size * 4 * 2) / 1e9:.2f} GB")
        print(f"MPS 已分配: {self._mem_allocated_gb():.2f} GB")

        print("🔥 预热中...")
        for _ in tqdm(range(10), desc="预热进度"):
            c = torch.bmm(a, b)
        self._sync()

        print(f"⏱️  开始 {duration} 秒测试...")
        start_time = time.time()
        iterations = 0
        total_flops = 0
        power_readings, util_readings = [], []

        pbar = tqdm(total=duration, desc="测试进度", unit="s")
        while time.time() - start_time < duration:
            c = torch.bmm(a, b)
            self._sync()
            iterations += 1
            total_flops += batch_size * 2 * size * size * size

            if iterations % 10 == 0:
                power, util = self.get_gpu_stats()
                power_readings.append(power)
                util_readings.append(util)

            elapsed = time.time() - start_time
            pbar.update(min(elapsed, duration) - pbar.n)
            pbar.set_postfix({
                'iter': iterations,
                '功耗': f'{power_readings[-1] if power_readings else 0:.0f}W',
                '利用率': f'{util_readings[-1] if util_readings else 0:.0f}%'
            })
        pbar.close()

        elapsed_time = time.time() - start_time
        avg_tflops = (total_flops / elapsed_time) / 1e12
        avg_power = np.mean(power_readings) if power_readings else 0
        avg_util = np.mean(util_readings) if util_readings else 0
        max_power = max(power_readings) if power_readings else 0
        max_util = max(util_readings) if util_readings else 0

        print("\n" + "=" * 70)
        print("📈 测试结果:")
        print(f"总迭代次数: {iterations}")
        print(f"总耗时: {elapsed_time:.2f} 秒")
        print(f"平均每次迭代耗时: {elapsed_time / iterations * 1000:.2f} ms")
        print(f"平均性能: {avg_tflops:.2f} TFLOPS")
        if power_readings:
            print(f"平均功耗: {avg_power:.1f} W")
            print(f"峰值功耗: {max_power:.1f} W")
        print(f"平均 GPU 利用率: {avg_util:.1f}%")
        print(f"峰值 GPU 利用率: {max_util:.1f}%")

        del a, b, c
        self._empty_cache()

        return {'tflops': avg_tflops, 'iterations': iterations, 'elapsed': elapsed_time,
                'avg_power': avg_power, 'max_power': max_power,
                'avg_util': avg_util, 'max_util': max_util}

    def test_matmul_single(self, size, duration=60):
        print(f"\n⚡ 单矩阵乘法测试（替代原多流测试，MPS 不支持用户级多流）")
        print(f"矩阵大小: {size}x{size}, 持续时间: {duration}s")
        print("-" * 70)

        a = torch.randn(size, size, dtype=torch.float32, device=self.device)
        b = torch.randn(size, size, dtype=torch.float32, device=self.device)
        print(f"输入矩阵显存: {(2 * size * size * 4) / 1e9:.2f} GB")
        print(f"MPS 已分配: {self._mem_allocated_gb():.2f} GB")

        for _ in tqdm(range(10), desc="预热进度"):
            c = torch.mm(a, b)
        self._sync()

        print(f"⏱️  开始 {duration} 秒测试...")
        start_time = time.time()
        iterations = 0
        total_flops = 0
        pbar = tqdm(total=duration, desc="测试进度", unit="s")

        while time.time() - start_time < duration:
            c = torch.mm(a, b)
            self._sync()
            iterations += 1
            total_flops += 2 * size * size * size
            elapsed = time.time() - start_time
            pbar.update(min(elapsed, duration) - pbar.n)
            pbar.set_postfix({'TFLOPS': f'{(total_flops / max(elapsed, 1e-9)) / 1e12:.2f}'})
        pbar.close()

        elapsed_time = time.time() - start_time
        avg_tflops = (total_flops / elapsed_time) / 1e12

        print("\n" + "=" * 70)
        print("📈 单矩阵测试结果:")
        print(f"总迭代次数: {iterations}")
        print(f"总耗时: {elapsed_time:.2f} 秒")
        print(f"平均性能: {avg_tflops:.2f} TFLOPS")

        del a, b, c
        self._empty_cache()
        return avg_tflops

    def run_full_test(self):
        optimal_size = self.find_optimal_matrix_size()

        # duration 可自行调小，方便快速试跑
        result1 = self.test_matmul_batch(size=min(optimal_size, 4096), batch_size=4, duration=120)
        result2 = self.test_matmul_batch(size=min(optimal_size, 2048), batch_size=16, duration=60)
        result3 = self.test_matmul_single(size=min(optimal_size, 4096), duration=60)

        print("\n" + "=" * 70)
        print("🏆 完整测试总结")
        print("=" * 70)
        print(f"测试1 (batch4, {min(optimal_size,4096)}): {result1['tflops']:.2f} TFLOPS")
        print(f"测试2 (batch16, {min(optimal_size,2048)}): {result2['tflops']:.2f} TFLOPS")
        print(f"测试3 (单矩阵, {min(optimal_size,4096)}): {result3:.2f} TFLOPS")

        best = max(result1['tflops'], result2['tflops'], result3)
        print(f"\n🎯 最佳性能: {best:.2f} TFLOPS")
        if self.peak_fp32:
            print(f"📊 相对理论峰值利用率: {best / self.peak_fp32 * 100:.1f}%")
        else:
            print("📊 未识别芯片，跳过理论峰值利用率计算")


if __name__ == "__main__":
    test = MacGPU_Ultimate_Test()
    test.run_full_test()

🚀 Mac GPU 终极性能测试
芯片: Apple M1
后端: mps
总内存: 8.6 GB
可用内存: 1.1 GB
FP32 理论峰值(估算): 2.6 TFLOPS
建议矩阵大小: 6656x6656

📊 批处理矩阵乘法测试
矩阵大小: 4096x4096, 批大小: 4, 持续时间: 120s
----------------------------------------------------------------------
输入矩阵显存: 0.54 GB
MPS 已分配: 0.54 GB
🔥 预热中...


预热进度: 100%|██████████| 10/10 [00:00<00:00, 62.15it/s]


⏱️  开始 120 秒测试...


测试进度: 100%|██████████| 120.0/120 [02:00<00:00,  1.00s/s, iter=137, 功耗=0W, 利用率=0%]             



📈 测试结果:
总迭代次数: 137
总耗时: 120.49 秒
平均每次迭代耗时: 879.49 ms
平均性能: 0.63 TFLOPS
平均功耗: 0.0 W
峰值功耗: 0.0 W
平均 GPU 利用率: 0.0%
峰值 GPU 利用率: 0.0%

📊 批处理矩阵乘法测试
矩阵大小: 2048x2048, 批大小: 16, 持续时间: 60s
----------------------------------------------------------------------
输入矩阵显存: 0.54 GB
MPS 已分配: 0.54 GB
🔥 预热中...


预热进度: 100%|██████████| 10/10 [00:00<00:00, 207.59it/s]


... (3 outputs left out)

预热进度: 100%|██████████| 10/10 [00:00<00:00, 310.23it/s]


⏱️  开始 60 秒测试...


测试进度: 100%|██████████| 60.0/60 [01:00<00:00,  1.00s/s, TFLOPS=0.73]               



📈 单矩阵测试结果:
总迭代次数: 320
总耗时: 60.16 秒
平均性能: 0.73 TFLOPS

🏆 完整测试总结
测试1 (batch4, 4096): 0.63 TFLOPS
测试2 (batch16, 2048): 0.75 TFLOPS
测试3 (单矩阵, 4096): 0.73 TFLOPS

🎯 最佳性能: 0.75 TFLOPS
📊 相对理论峰值利用率: 28.7%
